# 06. 도구 연동 (Tools Integration)

> 에이전트의 핵심은 **언제 어떤 도구를 호출할지** LLM이 직접 결정하는 것이에요. `@tool`, `bind_tools`, `ToolNode`, `tools_condition` 으로 챗봇 ↔ 도구 루프를 만드는 부품을 모두 다룹니다.

## 학습 목표

이 노트북을 마치면 다음을 할 수 있어요:

1. `@tool` 데코레이터로 Python 함수를 LLM이 호출할 수 있는 도구(Tool)로 변환할 수 있어요
2. `llm.bind_tools(tools)`로 LLM에 도구를 바인딩하고, `tool_calls`가 포함된 AIMessage 구조를 이해할 수 있어요
3. `BasicToolNode`를 직접 구현해서 도구 실행 노드의 내부 동작을 이해하고, `ToolNode`(사전 구축 컴포넌트)와 차이를 설명할 수 있어요
4. `tools_condition`과 `add_conditional_edges`로 챗봇 ↔ 도구 루프를 가진 에이전트 그래프를 구성할 수 있어요
5. `handle_tool_errors` 옵션으로 도구 실행 중 발생하는 에러를 처리할 수 있어요

## 사전 지식

- 이전 노트북 `05-ChatBot.ipynb`에서 배운 StateGraph, State, add_messages 개념
- LangGraph의 Node와 Edge 추가 방법 (`add_node`, `add_edge`)
- Python 데코레이터 기본 이해


## 도구 연동 에이전트의 구조

이전 노트북에서 만든 챗봇은 LLM이 학습한 지식만 활용할 수 있었어요. 하지만 실제 에이전트는 웹 검색, 계산기, 데이터베이스 조회 같은 **외부 도구(Tool)**를 호출해서 실시간 정보를 가져올 수 있어야 해요.

도구 연동의 핵심은 **조건부 루프(Conditional Loop)**예요. 이것은 마치 **도서관 사서**와 같아요. 질문을 받으면 머릿속 지식으로 바로 답할 수도 있지만, 정확한 정보가 필요하면 책(도구)을 찾아보고, 찾은 내용을 바탕으로 답변을 완성해요. 한 권으로 부족하면 다른 책도 참고하죠. 이 "판단 -> 조사 -> 답변" 반복이 에이전트의 핵심 루프예요. LLM이 도구 호출이 필요하다고 판단하면 도구를 실행하고 결과를 다시 LLM에 전달해요. 충분한 정보가 모이면 루프에서 벗어나 최종 답변을 생성하죠.

> 🔑 **핵심 개념**: 에이전트의 핵심은 **LLM이 다음 행동을 스스로 결정**한다는 거예요. `tool_calls`가 있으면 도구를 실행하고, 없으면 최종 답변을 생성해요. LangGraph는 이 루프를 그래프 구조로 표현해요.

### 전체 아키텍처

```mermaid
flowchart TD
    U(["사용자 입력<br>User Input"]) --> S(["__start__"])
    S --> C["chatbot 노드<br>LLM + bind_tools"]
    C -- "tool_calls 있음" --> T["tools 노드<br>ToolNode 실행"]
    C -- "tool_calls 없음" --> E(["__end__"])
    T -- "ToolMessage 반환" --> C
    E --> R(["최종 응답<br>Final Answer"])

    classDef input fill:#d4edda,stroke:#28a745,color:#155724
    classDef process fill:#cce5ff,stroke:#007bff,color:#004085
    classDef tools fill:#e2d5f1,stroke:#6f42c1,color:#3d1f6e
    classDef output fill:#fff3cd,stroke:#ffc107,color:#856404

    class U,S input
    class C process
    class T tools
    class E,R output
```

### 핵심 구성 요소

| 구성 요소 | 역할 | 핵심 API |
|-----------|------|----------|
| **@tool 데코레이터** | Python 함수 → LLM 호출 가능 도구로 변환 | `from langchain.tools import tool` |
| **bind_tools** | LLM에 사용 가능한 도구 목록 등록 | `llm.bind_tools(tools)` |
| **ToolNode** | tool_calls 기반으로 도구를 자동 실행하는 노드 | `from langgraph.prebuilt import ToolNode` |
| **tools_condition** | tool_calls 유무로 다음 노드를 결정하는 함수 | `from langgraph.prebuilt import tools_condition` |
| **add_conditional_edges** | 조건 함수 결과에 따라 분기 엣지 추가 | `graph_builder.add_conditional_edges(...)` |

### 메시지 흐름

```mermaid
sequenceDiagram
    participant U as 사용자
    participant C as chatbot 노드
    participant T as tools 노드

    U->>C: HumanMessage("오늘 날씨는?")
    C->>C: LLM 판단: 도구 필요
    C->>T: AIMessage(tool_calls=[search])
    T->>T: search 도구 실행
    T->>C: ToolMessage(결과)
    C->>C: LLM 판단: 충분한 정보
    C->>U: AIMessage(최종 답변)
```


## 환경 설정


In [3]:
# API 키를 환경변수로 관리하기 위한 설정
from dotenv import load_dotenv

# .env 파일에서 API 키 로드
load_dotenv(override=True)


True

In [ ]:
# LangSmith 추적 설정 (선택사항 - 실행 흐름을 LangSmith에서 시각화할 수 있어요)
import os

# LangSmith 추적 활성화: 에이전트의 도구 호출 흐름을 시각적으로 디버깅할 수 있어요
# os.environ["LANGCHAIN_TRACING_V2"] = "true"
# os.environ["LANGCHAIN_PROJECT"] = "LangGraph-V1-Tutorial-05-Tools"


## 1. @tool 데코레이터로 도구 정의하기

LangChain에서 도구(Tool)를 만드는 가장 쉬운 방법은 `@tool` 데코레이터예요. Python 함수에 데코레이터를 붙이면 LLM이 호출할 수 있는 도구로 자동 변환돼요.


In [4]:
from langchain.tools import tool

@tool
def multiply(a: int, b: int) -> int:
    """두 정수를 곱하고 반환한다. 곱셈 계산이 필요할 때 이 도구를 사용한다."""
    return a * b

@tool
def add(a: int, b: int) -> int:
    """두 정수를 더하고 반환한다. 덧셈 계산이 필요할 때 이 도구를 사용한다."""
    return a + b


@tool
def getWeather(city: str) -> str:
    """
    도시명을 받아서 해당 도시의 현재 날씨에 대해 반환한다.
    날씨는 {weather_data} 에서 검색해서 찾는다.

    Args:
        city: 날씨를 검색할 도시 이름
    """

    weather_data = {
        "서울": "맑음, 기온 18°C, 습도 55%",
        "부산": "흐림, 기온 21°C, 습도 70%",
        "제주": "비, 기온 23°C, 습도 85%",
    }
    # 등록된 도시면 날씨 반환, 없으면 기본 메시지
    return weather_data.get(city, f"{city}의 날씨 정보를 찾을 수 없어요. 서울, 부산, 제주 중 하나를 입력해주세요.")

tools = [multiply, add, getWeather]

tools[2].description


/Users/elixir/dev/lecture/hk-toss/10_langgraph/.venv/lib/python3.14/site-packages/langgraph/checkpoint/serde/encrypted.py:5: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


'도시명을 받아서 해당 도시의 현재 날씨에 대해 반환한다.\n날씨는 {weather_data} 에서 검색해서 찾는다.\n\nArgs:\n    city: 날씨를 검색할 도시 이름'

### 도구의 스키마 확인

`@tool` 데코레이터는 함수 시그니처와 docstring을 분석해서 LLM에 전달할 JSON 스키마를 자동으로 생성해요. LLM은 이 스키마를 보고 도구 호출 방법을 파악하죠.


In [5]:
import json

schema = getWeather.args_schema.model_json_schema()
print(json.dumps(schema, indent=2, ensure_ascii=False))

{
  "description": "도시명을 받아서 해당 도시의 현재 날씨에 대해 반환한다.\n날씨는 {weather_data} 에서 검색해서 찾는다.\n\nArgs:\n    city: 날씨를 검색할 도시 이름",
  "properties": {
    "city": {
      "title": "City",
      "type": "string"
    }
  },
  "required": [
    "city"
  ],
  "title": "getWeather",
  "type": "object"
}


## 2. LLM에 도구 바인딩하기 (bind_tools)

`bind_tools()` 메서드로 LLM에 사용 가능한 도구 목록을 등록해요. 도구가 바인딩된 LLM은 사용자 입력을 분석해서 도구 호출이 필요한지 스스로 판단해요.


In [6]:
from langchain.chat_models import init_chat_model

llm = init_chat_model("openai:gpt-4o-mini")

llm_with_tools = llm.bind_tools(tools)

In [7]:
from langchain.messages import HumanMessage

test_message = HumanMessage(content="서울 날씨 알려줘")
res = llm_with_tools.invoke([test_message])
res

AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 14, 'prompt_tokens': 166, 'total_tokens': 180, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_cba81caeca', 'id': 'chatcmpl-DfIj0zPPWZZUbG3W0EfQu9RV2rpLt', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019e24ea-5367-72c3-b0cf-b5669f29c743-0', tool_calls=[{'name': 'getWeather', 'args': {'city': '서울'}, 'id': 'call_79Z8YqmXQv60WJ6GeZpa38CX', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 166, 'output_tokens': 14, 'total_tokens': 180, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [12]:
from langchain.messages import HumanMessage

test_message = HumanMessage(content="안녕하세요")
res = llm_with_tools.invoke([test_message])
res

AIMessage(content='안녕하세요! 어떻게 도와드릴까요?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 11, 'prompt_tokens': 163, 'total_tokens': 174, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_cba81caeca', 'id': 'chatcmpl-DfI7CDXlUMc5DqwD7HNkZMDEzigOa', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019e24c6-94b1-7c00-83be-a017fe43c9f3-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 163, 'output_tokens': 11, 'total_tokens': 174, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

## 3. BasicToolNode 직접 구현하기

앞서 `bind_tools()`로 LLM이 **어떤 도구를 호출할지 결정**하는 방법을 배웠어요. 하지만 LLM은 도구를 직접 실행하지 않아요. 누군가가 `tool_calls` 정보를 받아서 실제로 함수를 호출해야 하죠. 그 역할을 하는 것이 **도구 실행 노드**예요.

이 섹션에서는 도구 실행 노드를 **직접 구현**해서 내부 동작을 이해해볼게요. LangGraph가 제공하는 `ToolNode`를 바로 쓰기 전에, 어떻게 작동하는지 먼저 파악하는 거예요.


In [10]:
import json
from langchain.messages import ToolMessage

class BasicToolNode:

    # 툴 노드는 도구함처럼 생각 해야 한다. 따라서 tools를 받아낸다.
    def __init__(self, tools: list) -> None:
        print("=========", { tool.name : tool for tool in tools }, "=========")
        self.tools_by_name = { tool.name : tool for tool in tools }

    def __call__(self, state: dict) -> dict:
        messages = state.get("messages", [])
        if not messages:
            raise ValueError("메시지가 없습니다.")

        ai_message = messages[-1]

        tool_results = []

        for tool_call in ai_message.tool_calls:
            tool_name = tool_call["name"]
            tool_args = tool_call["args"]
            tool_call_id = tool_call["id"]

            tool_to_run = self.tools_by_name.get(tool_name)

            if not tool_to_run:
                raise ValueError(f"{tool_name} 없음")

            result = tool_to_run.invoke(tool_args)
            tool_results.append(
                ToolMessage(
                    content = str(result),
                    name = tool_name,
                    tool_call_id = tool_call_id
                )
            )

        return {"messages": tool_results}


In [11]:
from langchain.messages import AIMessage

# ---------------------------------------------------
# BasicToolNode 직접 테스트
# ---------------------------------------------------
# 실제로는 LLM이 생성하지만, 테스트를 위해 수동으로 AIMessage를 만들어요
test_ai_message = AIMessage(
    content="",  # 도구를 호출할 때는 content가 비어있어요
    tool_calls=[
        {
            "name": "multiply",     # 호출할 도구 이름
            "args": {"a": 15, "b": 7},  # 도구 인자
            "id": "test_call_001",  # 고유 호출 ID
            "type": "tool_call",
        }
    ],
)

# BasicToolNode 인스턴스 생성 및 실행
basic_tool_node = BasicToolNode(tools)
result = basic_tool_node({"messages": [test_ai_message]})

# === BasicToolNode 실행 결과 ===
for msg in result["messages"]:
    print(f"메시지 타입: {type(msg).__name__}")
    print(f"도구 이름: {msg.name}")
    print(f"결과: {msg.content}")
    print(f"호출 ID: {msg.tool_call_id}")

========= {'multiply': StructuredTool(name='multiply', description='두 정수를 곱하고 반환한다. 곱셈 계산이 필요할 때 이 도구를 사용한다.', args_schema=<class 'langchain_core.utils.pydantic.multiply'>, func=<function multiply at 0x1111e9f30>), 'add': StructuredTool(name='add', description='두 정수를 더하고 반환한다. 덧셈 계산이 필요할 때 이 도구를 사용한다.', args_schema=<class 'langchain_core.utils.pydantic.add'>, func=<function add at 0x111e000f0>), 'getWeather': StructuredTool(name='getWeather', description='도시명을 받아서 해당 도시의 현재 날씨에 대해 반환한다.\n날씨는 {weather_data} 에서 검색해서 찾는다.\n\nArgs:\n    city: 날씨를 검색할 도시 이름', args_schema=<class 'langchain_core.utils.pydantic.getWeather'>, func=<function getWeather at 0x111dd3950>)} =========
메시지 타입: ToolMessage
도구 이름: multiply
결과: 105
호출 ID: test_call_001


## 4. 조건부 엣지 (Conditional Edge)와 에이전트 그래프 구축

도구 노드가 준비되었으니, 이제 챗봇과 도구가 연결된 완전한 에이전트 그래프를 만들어볼게요.

핵심은 **조건부 엣지(Conditional Edge)**예요. 일반 엣지는 항상 같은 노드로 이동하지만, 조건부 엣지는 상태에 따라 다른 노드로 분기해요.


### 4-1. 라우터 함수 직접 구현 (이해용)

먼저 `tools_condition`이 내부에서 어떻게 동작하는지 이해하기 위해 라우터 함수를 직접 구현해볼게요.


In [ ]:
from typing import Annotated, TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.graph import add_messages

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 상태 정의 (이전 챗봇과 동일)
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


### 4-2. 에이전트 그래프 구축 (BasicToolNode 사용)

직접 구현한 `BasicToolNode`와 라우터 함수로 에이전트 그래프를 구축해볼게요.


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 챗봇 노드 정의 (도구 바인딩된 LLM 사용)
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
from IPython.display import Image, display

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 그래프 흐름: START → chatbot → tools → chatbot → ... → END
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 에이전트 실행: 도구 호출이 필요한 질문
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 에이전트 실행: 도구 호출이 필요 없는 질문 (바로 END로 이동)
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


## 5. ToolNode와 tools_condition (사전 구축 컴포넌트)

앞서 `BasicToolNode`와 `route_tools`를 직접 구현했어요. LangGraph는 이 두 가지를 사전 구축(pre-built) 컴포넌트로 제공해요. 실제 프로젝트에서는 이 컴포넌트를 사용하면 코드가 훨씬 간결해져요.

| 직접 구현 | 사전 구축 컴포넌트 | 차이 |
|-----------|------------------|------|
| `BasicToolNode` | `ToolNode` | 에러 처리, 병렬 실행, 상태 주입 등 고급 기능 |
| `route_tools()` | `tools_condition` | 동일한 로직, 이미 검증된 구현 |


In [ ]:
from langgraph.prebuilt import ToolNode, tools_condition

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: LangGraph 사전 구축 컴포넌트 가져오기
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: ToolNode + tools_condition으로 간결하게 그래프 구성
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
from IPython.display import Image, display

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 그래프 흐름: START → chatbot → tools → chatbot → ... → END
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 사전 구축 컴포넌트 그래프 실행: 날씨 + 계산 복합 질문
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


## 5-1. 실전 도구: TavilySearch로 웹 검색

지금까지 더미 도구(get_weather)로 도구 실행 흐름을 이해했어요. 이제 **실제로 웹 검색을 수행하는 도구**를 연결해볼게요.

`TavilySearch`는 AI 에이전트를 위해 최적화된 검색 API예요. LangChain V1에서는 `langchain-tavily` 패키지로 바로 사용할 수 있어요.


In [ ]:
from langchain_tavily import TavilySearch

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: TavilySearch: AI 에이전트용 웹 검색 도구
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
real_llm = init_chat_model("openai:gpt-4o-mini")

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 실전 에이전트: TavilySearch + 계산 도구
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


## 6. 도구 에러 처리 (handle_tool_errors)

`ToolNode`는 도구 실행 중 발생하는 에러를 자동으로 처리할 수 있어요. 기본적으로 `handle_tool_errors=True`가 설정되어 있어서, 에러가 발생하면 `ToolMessage`에 에러 메시지를 담아 LLM에 전달해요.

### handle_tool_errors 옵션 정리

| 설정값 | 동작 |
|--------|------|
| `True` (기본값) | 모든 에러를 잡아 에러 메시지를 LLM에 전달 |
| `False` | 에러 처리 비활성화, 예외가 그대로 발생 |
| `"커스텀 메시지"` | 에러 발생 시 지정된 문자열을 LLM에 전달 |
| `callable` | 에러 객체를 받아 커스텀 메시지를 반환하는 함수 |
| `(ValueError, TypeError)` | 지정된 예외 타입만 처리 |


In [ ]:
from langgraph.graph import MessagesState

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 에러를 발생시키는 테스트 도구 정의
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 에러 처리 동작 테스트: 0으로 나누기
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


## 7. 병렬 도구 호출

`ToolNode`는 여러 도구를 동시에 호출하는 **병렬 도구 호출**도 지원해요. AIMessage의 `tool_calls` 리스트에 여러 도구를 담으면, `ToolNode`가 모두 실행하고 각각의 `ToolMessage`를 반환해요.


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 병렬 도구 호출 테스트
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


## 8. ToolRuntime으로 그래프 상태 접근하기

도구 실행 중에 그래프의 현재 상태에 접근해야 하는 경우가 있어요. 예를 들어, 현재까지의 대화 메시지 수를 확인하거나 사용자 설정 값을 참조할 때예요.

`ToolRuntime`을 도구의 매개변수로 선언하면, `ToolNode`가 자동으로 현재 상태를 주입해줘요. **LLM에게는 이 매개변수가 노출되지 않으므로** 도구 호출 방식에 영향을 주지 않아요.


In [ ]:
from langchain.tools import ToolRuntime

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: ToolRuntime을 사용한 상태 접근 도구
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
from langchain.messages import HumanMessage, AIMessage

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: ToolRuntime 상태 주입 테스트
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


## 9. 실습: 나만의 도구를 만들어서 에이전트에 연결하기

지금까지 배운 내용을 바탕으로 나만의 도구를 만들고, 에이전트 그래프에 연결해보세요.


In [ ]:
from langchain.tools import tool
from langchain.chat_models import init_chat_model
from langgraph.graph import StateGraph, START, END, add_messages
from langgraph.prebuilt import ToolNode, tools_condition
from typing import Annotated, TypedDict
from IPython.display import Image, display

practice_llm = init_chat_model("openai:gpt-4o-mini")


# ============================================================
# TODO: 아래에서 나만의 도구를 정의해보세요!
# 힌트: @tool 데코레이터를 사용하고 docstring을 명확하게 작성하세요
#       예: 온도 변환 (섭씨 ↔ 화씨), 텍스트 길이 계산, 대소문자 변환 등
# 예상 결과: LLM이 질문을 분석해서 적절한 도구를 자동으로 선택해요
# ============================================================


# TODO: 여기에 두 번째 도구를 추가해보세요!
# ============================================================


# ---------------------------------------------------
# 실습 도구들로 에이전트 그래프 구성
# ---------------------------------------------------
# 상태 정의


# TODO: 아래 질문을 수정해서 나만의 도구가 잘 작동하는지 확인해보세요!
# ============================================================


## 핵심 요약

이 노트북에서 다음 내용을 배웠어요:

- **@tool 데코레이터**: Python 함수를 LLM이 호출 가능한 도구로 변환해요. docstring이 LLM의 도구 선택 가이드가 되므로 명확하게 작성하는 게 중요해요
- **bind_tools()**: LLM에 사용 가능한 도구를 등록해요. 바인딩된 LLM은 `tool_calls`가 담긴 AIMessage를 반환할 수 있어요
- **BasicToolNode vs ToolNode**: `BasicToolNode`를 직접 구현해서 도구 실행 원리를 이해했어요. 실제로는 `ToolNode`를 사용하면 에러 처리, 병렬 실행 등 고급 기능을 쉽게 활용할 수 있어요
- **tools_condition**: `AIMessage`에 `tool_calls`가 있으면 `"tools"`, 없으면 `END`를 반환하는 사전 구축 라우터 함수예요
- **add_conditional_edges**: 조건 함수 결과에 따라 분기하는 엣지를 추가해요. `chatbot ↔ tools` 루프를 형성하는 핵심이에요
- **handle_tool_errors**: `ToolNode`의 에러 처리 옵션으로, `True`/`False`/문자열/함수 등 다양한 방식으로 에러를 핸들링할 수 있어요
- **ToolRuntime**: 도구 파라미터로 선언하면 `ToolNode`가 현재 그래프 상태를 자동으로 주입해줘요. LLM에게는 노출되지 않아요


## 다음 노트북 예고

다음 `07-Memory-Checkpointer.ipynb`에서는 **InMemorySaver와 thread_id**를 배워요.

지금까지 만든 에이전트는 실행이 끝나면 대화 내용을 기억하지 못했어요. 다음 노트북에서는 `InMemorySaver`로 체크포인트를 설정하고 `thread_id`로 대화 세션을 구분해서, 이전 대화를 기억하는 **메모리가 있는 에이전트**를 만들어볼 거예요!
